# PARAGON - Stage 1 (Mid-Semester Review)

Runs the full pilot pipeline on Kaggle GPU: dataset -> DIPPER paraphrases -> detectors -> evaluation.

**Prerequisites:**
1. Edit `REPO_URL` in the next cell to point at your GitHub repo.
2. (Optional) Add a Kaggle Secret named `GPTZERO_API_KEY` to enable the GPTZero detector.
3. Enable GPU accelerator (T4 x2).

Outputs appear under `/kaggle/working/paragon/results/` and must be downloaded to the repo.

In [ ]:
REPO_URL = "https://github.com/kushalk-hub/MVGR-AI.git"
WORK_DIR = "/kaggle/working/paragon"


In [ ]:
import os
import subprocess

BASE = "/kaggle/working"
os.makedirs(BASE, exist_ok=True)
subprocess.run(["rm", "-rf", "paragon"], cwd=BASE)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "paragon"], cwd=BASE, check=True)
os.chdir(os.path.join(BASE, "paragon"))
print(os.getcwd())
print(sorted(os.listdir(".")))


In [ ]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "nltk", "tabulate"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
print("install complete")

def _ver(name):
    try:
        mod = __import__(name)
        return getattr(mod, "__version__", "?")
    except Exception as e:
        return f"MISSING ({type(e).__name__})"

print("transformers:", _ver("transformers"))
print("bitsandbytes:", _ver("bitsandbytes"))
print("accelerate:", _ver("accelerate"))


In [ ]:
!python data/prepare_pilot.py


In [ ]:
!python paraphrase/dipper_generate.py


In [ ]:
!python paraphrase/qc.py


In [ ]:
# --detectors roberta binoculars always; gptzero uses the limit (set higher if you have quota)
!python detectors/run_pilot.py --detectors roberta binoculars gptzero --gptzero-limit 80


In [ ]:
!python detectors/calibrate_binoculars.py


In [ ]:
!python evaluation/evaluate.py


## Download results

Everything you need lives in `/kaggle/working/paragon/`:
- `results/pilot_scores.csv` - scored CSV
- `results/pilot_scores_eval.csv` - eval-split scores
- `results/degradation_plot.png` - headline chart
- `results/results_table.csv` - AUROC / TPR@1% / F1 per level
- `results/summary.md` - auto summary

Download these files and commit them back into your repo.